### Zero shot text classification for top 10000 articles to see whether they are correctly identified as sports/non-sports

In [1]:
pip install transformers pandas tqdm torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm.notebook import tqdm
import json

c:\Users\liqiq\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "facebook/bart-large-mnli"
DEVICE = 0 if torch.cuda.is_available() else -1

print(f"Loading model: {MODEL_NAME} on device: {'GPU' if DEVICE == 0 else 'CPU'}")

# pipeline is a function from HuggingFace's transformers library
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=DEVICE,
    batch_size=32
)

classifier

Loading model: facebook/bart-large-mnli on device: CPU


Device set to use cpu


In [4]:
df = pd.read_json("entity_results_after.jsonl",lines=True)

df

,QID,status,label,description,attributes,error_message
0,Q6876,success,volleyball at the Summer Olympics,No description found,{'Commons category': 'Volleyball at the Olympi...,NaN
1,Q1083432,success,Kieran Trippier,English association football player (born 1990),"{'occupation': 'association football player', ...",NaN
2,Q10479625,success,Qiang Hui,Chinese military general from the state of Qin...,"{'instance of': 'human', 'occupation': 'milita...",NaN
3,Q1093205,success,cisgender,gender identity descriptor,"{'instance of': 'gender identity', 'Freebase I...",NaN
4,Q715193,success,Hajime no Ippo,Japanese manga series,{'MusicBrainz release group ID': '1db8fa91-16f...,NaN
...,...,...,...,...,...,...
16383,Q13590412,success,Prince George of Wales,"British prince, child of William, Prince of Wa...","{'mother': 'Catherine, Princess of Wales', 'fa...",NaN
16384,Q48444,success,phoenix,long-lived bird that is cyclically regenerated...,{'Commons category': 'Phoenix (mythical bird)'...,NaN
16385,Q483837,success,Luka Modrić,Croatian association football player (born 1985),"{'sex or gender': 'male', 'member of sports te...",NaN
16386,Q112014,success,Cabinet of the United Kingdom,collective decision-making body of the British...,{'Commons category': 'Members of the Cabinet o...,NaN


In [5]:
# Use .apply with a lambda function
df['occupation'] = df['attributes'].apply(lambda x: x.get('occupation') if isinstance(x, dict) else None)
df.head(10)

,QID,status,label,description,attributes,error_message,occupation
0,Q6876,success,volleyball at the Summer Olympics,No description found,{'Commons category': 'Volleyball at the Olympi...,NaN,None
1,Q1083432,success,Kieran Trippier,English association football player (born 1990),"{'occupation': 'association football player', ...",NaN,association football player
2,Q10479625,success,Qiang Hui,Chinese military general from the state of Qin...,"{'instance of': 'human', 'occupation': 'milita...",NaN,military commander
3,Q1093205,success,cisgender,gender identity descriptor,"{'instance of': 'gender identity', 'Freebase I...",NaN,None
4,Q715193,success,Hajime no Ippo,Japanese manga series,{'MusicBrainz release group ID': '1db8fa91-16f...,NaN,None
5,Q13476175,success,Lorde,"Croatian-New Zealand singer, songwriter and pr...",{'MusicBrainz artist ID': '8e494408-8620-4c6a-...,NaN,singer-songwriter
6,Q35610,success,Arthur Conan Doyle,British writer and physician (1859–1930),"{'honorific prefix': 'Sir', 'image': 'Arthur C...",NaN,physician
7,Q131120,success,Anna Kournikova,Russian tennis player and model,"{'Commons category': 'Anna Kournikova', 'occup...",NaN,tennis player
8,Q507338,success,Nikkei 225,Japanese stock market index,"{'Commons category': 'Nikkei 225', 'topic's ma...",NaN,None
9,Q116369841,success,Macklin Celebrini,Canadian ice hockey player (born 2006),"{'instance of': 'human', 'sex or gender': 'mal...",NaN,ice hockey player


In [ ]:
df["text_for_clf"] = (df["label"].fillna("") + ". " +  df["occupation"].fillna("") + ". " +
    df["description"].fillna("") ) 

In [7]:
s_labels = ["Sport", "Non-Sport"]

In [8]:
results = classifier(
        df["text_for_clf"].to_list(),
        candidate_labels=s_labels,
        hypothesis_template= "This text is about {}.",
        multi_label=False
    )

results

[{'sequence': 'volleyball at the Summer Olympics. . No description found',
  'labels': ['Sport', 'Non-Sport'],
  'scores': [0.8562607765197754, 0.1437392383813858]},
 {'sequence': 'Kieran Trippier. association football player. English association football player (born 1990)',
  'labels': ['Sport', 'Non-Sport'],
  'scores': [0.9273813366889954, 0.07261864095926285]},
 {'sequence': 'Qiang Hui. military commander. Chinese military general from the state of Qin during the Warring States period',
  'labels': ['Non-Sport', 'Sport'],
  'scores': [0.8045715689659119, 0.19542841613292694]},
 {'sequence': 'cisgender. . gender identity descriptor',
  'labels': ['Non-Sport', 'Sport'],
  'scores': [0.754611074924469, 0.2453889101743698]},
 {'sequence': 'Hajime no Ippo. . Japanese manga series',
  'labels': ['Non-Sport', 'Sport'],
  'scores': [0.8042069673538208, 0.1957930028438568]},
 {'sequence': 'Lorde. singer-songwriter. Croatian-New Zealand singer, songwriter and producer (born 1996)',
  'label

In [34]:
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

In [35]:
df['sport'] = df['attributes'].apply(lambda x: x.get('sport') if isinstance(x, dict) else None)
df.head(10)

,QID,status,label,description,attributes,error_message,occupation,text_for_clf,sport
0,Q6876,success,volleyball at the Summer Olympics,No description found,{'Commons category': 'Volleyball at the Olympi...,NaN,None,volleyball at the Summer Olympics. . No descri...,volleyball
1,Q1083432,success,Kieran Trippier,English association football player (born 1990),"{'occupation': 'association football player', ...",NaN,association football player,Kieran Trippier. association football player. ...,association football
2,Q10479625,success,Qiang Hui,Chinese military general from the state of Qin...,"{'instance of': 'human', 'occupation': 'milita...",NaN,military commander,Qiang Hui. military commander. Chinese militar...,None
3,Q1093205,success,cisgender,gender identity descriptor,"{'instance of': 'gender identity', 'Freebase I...",NaN,None,cisgender. . gender identity descriptor,None
4,Q715193,success,Hajime no Ippo,Japanese manga series,{'MusicBrainz release group ID': '1db8fa91-16f...,NaN,None,Hajime no Ippo. . Japanese manga series,None
5,Q13476175,success,Lorde,"Croatian-New Zealand singer, songwriter and pr...",{'MusicBrainz artist ID': '8e494408-8620-4c6a-...,NaN,singer-songwriter,Lorde. singer-songwriter. Croatian-New Zealand...,None
6,Q35610,success,Arthur Conan Doyle,British writer and physician (1859–1930),"{'honorific prefix': 'Sir', 'image': 'Arthur C...",NaN,physician,Arthur Conan Doyle. physician. British writer ...,cricket
7,Q131120,success,Anna Kournikova,Russian tennis player and model,"{'Commons category': 'Anna Kournikova', 'occup...",NaN,tennis player,Anna Kournikova. tennis player. Russian tennis...,tennis
8,Q507338,success,Nikkei 225,Japanese stock market index,"{'Commons category': 'Nikkei 225', 'topic's ma...",NaN,None,Nikkei 225. . Japanese stock market index,None
9,Q116369841,success,Macklin Celebrini,Canadian ice hockey player (born 2006),"{'instance of': 'human', 'sex or gender': 'mal...",NaN,ice hockey player,Macklin Celebrini. ice hockey player. Canadian...,ice hockey


In [36]:
# Select only three columns
df_new = df[["QID","text_for_clf", "sport"]].copy()

df_new.head(10)


,QID,text_for_clf,sport
0,Q6876,volleyball at the Summer Olympics. . No descri...,volleyball
1,Q1083432,Kieran Trippier. association football player. ...,association football
2,Q10479625,Qiang Hui. military commander. Chinese militar...,None
3,Q1093205,cisgender. . gender identity descriptor,None
4,Q715193,Hajime no Ippo. . Japanese manga series,None
5,Q13476175,Lorde. singer-songwriter. Croatian-New Zealand...,None
6,Q35610,Arthur Conan Doyle. physician. British writer ...,cricket
7,Q131120,Anna Kournikova. tennis player. Russian tennis...,tennis
8,Q507338,Nikkei 225. . Japanese stock market index,None
9,Q116369841,Macklin Celebrini. ice hockey player. Canadian...,ice hockey


In [37]:
df_new['truth'] = df_new['sport'].apply(lambda x: 'S' if x is not None else 'NS')

df_new.head(10)

,QID,text_for_clf,sport,truth
0,Q6876,volleyball at the Summer Olympics. . No descri...,volleyball,S
1,Q1083432,Kieran Trippier. association football player. ...,association football,S
2,Q10479625,Qiang Hui. military commander. Chinese militar...,None,NS
3,Q1093205,cisgender. . gender identity descriptor,None,NS
4,Q715193,Hajime no Ippo. . Japanese manga series,None,NS
5,Q13476175,Lorde. singer-songwriter. Croatian-New Zealand...,None,NS
6,Q35610,Arthur Conan Doyle. physician. British writer ...,cricket,S
7,Q131120,Anna Kournikova. tennis player. Russian tennis...,tennis,S
8,Q507338,Nikkei 225. . Japanese stock market index,None,NS
9,Q116369841,Macklin Celebrini. ice hockey player. Canadian...,ice hockey,S


In [42]:
df_new[df_new['truth']=='S']

,QID,text_for_clf,sport,truth
0,Q6876,volleyball at the Summer Olympics. . No descri...,volleyball,S
1,Q1083432,Kieran Trippier. association football player. ...,association football,S
6,Q35610,Arthur Conan Doyle. physician. British writer ...,cricket,S
7,Q131120,Anna Kournikova. tennis player. Russian tennis...,tennis,S
9,Q116369841,Macklin Celebrini. ice hockey player. Canadian...,ice hockey,S
...,...,...,...,...
16362,Q15531382,Katarzyna Niewiadoma. sport cyclist. Polish pr...,road bicycle racing,S
16365,Q1865564,Tommy Fleetwood. golfer. English professional ...,golf,S
16373,Q374239,Tyler Mane. actor. Canadian professional wrestler,professional wrestling,S
16378,Q525002,Troy Aikman. television presenter. American pr...,American football,S


In [14]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [43]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score

In [44]:
def evaluate_zero_shot(classification_results, dataframe):
    """
    Calculates and prints classification metrics based on zero-shot results.

    Args:
        classification_results (list): The list of dictionaries from the HF pipeline output.
        dataframe: included sport attribute to see whether it predicts the right one
    """

    labels = {"Sport": 'S', "Non-Sport": 'NS'}
    prediction_mapping = labels

    # Get the top predicted label and map it to 'S' or 'NS'
    y_pred = [prediction_mapping[result['labels'][0]] for result in classification_results]

    # Read the ground truth labels from the dataframe
    df_gt = dataframe
    y_true = df_gt['truth'].tolist()

    # Use 'S' (Sport) as the positive label for binary metrics (recall, precision, F1)

    # Confusion Matrix (labels=[Positive, Negative])
    # By default, sklearn sorts labels alphabetically. We force the order [S, NS] for clarity.
    cm = confusion_matrix(y_true, y_pred, labels=['S', 'NS'])

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # Recall (Sensitivity/True Positive Rate)
    recall = recall_score(y_true, y_pred, pos_label='S')

    # Precision (Positive Predictive Value)
    precision = precision_score(y_true, y_pred, pos_label='S')

    # F1-Score (Harmonic mean of precision and recall)
    f1 = f1_score(y_true, y_pred, pos_label='S')

    # Print Results
    print("--- Classification Evaluation ---")
    print(f"Confusion Matrix (True Labels: ['S', 'NS']):\n{cm}")
    print("\n")
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"Recall (for 'S'): {recall:.4f}")
    print(f"Precision (for 'S'): {precision:.4f}")
    print(f"F1-Score (for 'S'): {f1:.4f}")

In [45]:
evaluate_zero_shot(results, df_new)

--- Classification Evaluation ---
Confusion Matrix (True Labels: ['S', 'NS']):
[[ 3783   319]
 [  519 11767]]



Accuracy: 0.9489
Recall (for 'S'): 0.9222
Precision (for 'S'): 0.8794
F1-Score (for 'S'): 0.9003


##### True positive 3783
##### False positive 519 Is sport predicted NS
##### In total is 4302
##### True Negative 11767
##### False Negative 319 Not Sport predicted S

In [18]:
with open('results.json', 'r', encoding='utf-8') as f:
  t_results = json.load(f)

df_r = pd.DataFrame(t_results)
df_r.head()

,sequence,labels,scores
0,volleyball at the Summer Olympics. . No descri...,"[Sport, Non-Sport]","[0.8562607765197754, 0.1437392383813858]"
1,Kieran Trippier. association football player. ...,"[Sport, Non-Sport]","[0.9273813366889954, 0.07261864095926285]"
2,Qiang Hui. military commander. Chinese militar...,"[Non-Sport, Sport]","[0.8045715689659119, 0.19542841613292694]"
3,cisgender. . gender identity descriptor,"[Non-Sport, Sport]","[0.754611074924469, 0.2453889101743698]"
4,Hajime no Ippo. . Japanese manga series,"[Non-Sport, Sport]","[0.8042069673538208, 0.1957930028438568]"


In [19]:
df_r['predicted_label'] = df_r['labels'].apply(lambda x: x[0])
df_r.head()

,sequence,labels,scores,predicted_label
0,volleyball at the Summer Olympics. . No descri...,"[Sport, Non-Sport]","[0.8562607765197754, 0.1437392383813858]",Sport
1,Kieran Trippier. association football player. ...,"[Sport, Non-Sport]","[0.9273813366889954, 0.07261864095926285]",Sport
2,Qiang Hui. military commander. Chinese militar...,"[Non-Sport, Sport]","[0.8045715689659119, 0.19542841613292694]",Non-Sport
3,cisgender. . gender identity descriptor,"[Non-Sport, Sport]","[0.754611074924469, 0.2453889101743698]",Non-Sport
4,Hajime no Ippo. . Japanese manga series,"[Non-Sport, Sport]","[0.8042069673538208, 0.1957930028438568]",Non-Sport


In [20]:
df_r['score'] = df_r['scores'].apply(lambda x: x[0])
df_r.head()

,sequence,labels,scores,predicted_label,score
0,volleyball at the Summer Olympics. . No descri...,"[Sport, Non-Sport]","[0.8562607765197754, 0.1437392383813858]",Sport,0.856261
1,Kieran Trippier. association football player. ...,"[Sport, Non-Sport]","[0.9273813366889954, 0.07261864095926285]",Sport,0.927381
2,Qiang Hui. military commander. Chinese militar...,"[Non-Sport, Sport]","[0.8045715689659119, 0.19542841613292694]",Non-Sport,0.804572
3,cisgender. . gender identity descriptor,"[Non-Sport, Sport]","[0.754611074924469, 0.2453889101743698]",Non-Sport,0.754611
4,Hajime no Ippo. . Japanese manga series,"[Non-Sport, Sport]","[0.8042069673538208, 0.1957930028438568]",Non-Sport,0.804207


In [21]:
df_r = df_r.drop(columns=['labels', 'scores'])
df_r.head()

,sequence,predicted_label,score
0,volleyball at the Summer Olympics. . No descri...,Sport,0.856261
1,Kieran Trippier. association football player. ...,Sport,0.927381
2,Qiang Hui. military commander. Chinese militar...,Non-Sport,0.804572
3,cisgender. . gender identity descriptor,Non-Sport,0.754611
4,Hajime no Ippo. . Japanese manga series,Non-Sport,0.804207


In [22]:
df_new.head()

,QID,text_for_clf,sport,truth
0,Q6876,volleyball at the Summer Olympics. . No descri...,volleyball,S
1,Q1083432,Kieran Trippier. association football player. ...,association football,S
2,Q10479625,Qiang Hui. military commander. Chinese militar...,None,NS
3,Q1093205,cisgender. . gender identity descriptor,None,NS
4,Q715193,Hajime no Ippo. . Japanese manga series,None,NS


In [23]:
df_combined = pd.merge(df_r, df_new, left_on='sequence', right_on='text_for_clf')
df_combined.drop(columns=['sequence'], inplace=True)
df_combined.head()

,predicted_label,score,QID,text_for_clf,sport,truth
0,Sport,0.856261,Q6876,volleyball at the Summer Olympics. . No descri...,volleyball,S
1,Sport,0.927381,Q1083432,Kieran Trippier. association football player. ...,association football,S
2,Non-Sport,0.804572,Q10479625,Qiang Hui. military commander. Chinese militar...,None,NS
3,Non-Sport,0.754611,Q1093205,cisgender. . gender identity descriptor,None,NS
4,Non-Sport,0.804207,Q715193,Hajime no Ippo. . Japanese manga series,None,NS


In [24]:
df_sport = df_combined[["predicted_label","QID","score", "truth"]].copy()
df_sport.head(10)

,predicted_label,QID,score,truth
0,Sport,Q6876,0.856261,S
1,Sport,Q1083432,0.927381,S
2,Non-Sport,Q10479625,0.804572,NS
3,Non-Sport,Q1093205,0.754611,NS
4,Non-Sport,Q715193,0.804207,NS
5,Non-Sport,Q13476175,0.838603,NS
6,Non-Sport,Q35610,0.838213,S
7,Sport,Q131120,0.957127,S
8,Non-Sport,Q507338,0.667714,NS
9,Sport,Q116369841,0.982886,S


In [25]:
df_sport.to_csv("sport.csv",index=False)

In [46]:
false_positives_df = df_combined[(df_combined['predicted_label'] == 'Sport') & (df_combined['truth'] == 'NS')]
false_positives_df.head(10)

,predicted_label,score,QID,text_for_clf,sport,truth
14,Sport,0.813693,Q180672,list of IOC country codes. . country codes use...,None,NS
24,Sport,0.534777,Q1140085,Crimson Tide. . 1995 film by Tony Scott,None,NS
75,Sport,0.502642,Q1999739,Wolf ticket. . document with restrictive clauses,None,NS
81,Sport,0.987712,Q116859491,Mexico at the 2024 Summer Olympics. . sporting...,None,NS
107,Sport,0.786686,Q123751791,Individual Neutral Athletes at the 2024 Summer...,None,NS
146,Sport,0.717675,Q171401,futsal. . football-based game played on a hard...,None,NS
183,Sport,0.980402,Q125817343,UFC 304. . UFC mixed martial arts event in 2024,None,NS
210,Sport,0.652345,Q207770,Pirelli. . Italian multinational tyre manufact...,None,NS
280,Sport,0.996071,Q11828878,Przemysław Babiarz. sports journalist. Polish ...,None,NS
288,Sport,0.506971,Q722001,Luke Evans. film actor. Welsh actor and singer,None,NS


In [27]:
false_positives_df.shape[0]

519

In [28]:
false_negatives_df = df_combined[(df_combined['predicted_label'] == 'Non-Sport') & (df_combined['truth'] == 'S')]
false_negatives_df.head(10)

,predicted_label,score,QID,text_for_clf,sport,truth
6,Non-Sport,0.838213,Q35610,Arthur Conan Doyle. physician. British writer ...,cricket,S
16,Non-Sport,0.850996,Q5809,Che Guevara. politician. Argentine Marxist rev...,rugby union,S
56,Non-Sport,0.829601,Q186492,George S. Patton. autobiographer. United State...,competitive swimming,S
133,Non-Sport,0.818351,Q41529365,Stephen Paddock. mass murderer. American mass ...,poker,S
270,Non-Sport,0.639245,Q3064656,No label found. middle-distance runner. Kenyan...,athletics,S
346,Non-Sport,0.729628,Q710128,Jan-Michael Vincent. actor. American actor (19...,surfing,S
465,Non-Sport,0.686690,Q15615,Lil Wayne. rapper. American rapper,skateboarding,S
503,Non-Sport,0.880213,Q973475,"Dustin Diamond. actor. American actor, musicia...",professional wrestling,S
790,Non-Sport,0.793090,Q335880,Taylor Kitsch. actor. Canadian actor and model,ice hockey,S
808,Non-Sport,0.894024,Q485365,"Bear Grylls. explorer. English adventurer, wri...",mountaineering,S


In [29]:
false_negatives_df.shape[0]

366

In [84]:
import pandas as pd
all = pd.read_csv("data/d_all.csv")
all.head()

,date,country_code,project,article,qid,pageviews,language,language_full
0,2024-06-26,PE,es.wikipedia,Universo,Q1,90,es,Spanish
1,2024-06-26,FR,fr.wikipedia,Univers,Q1,98,fr,French
2,2024-06-26,BR,pt.wikipedia,Universo,Q1,141,pt,Portuguese
3,2024-06-26,AU,en.wikipedia,Universe,Q1,98,en,English
4,2024-06-26,JP,ja.wikipedia,宇宙,Q1,360,ja,Japanese


In [85]:
df_articles = all[["date","qid", "article","pageviews","language_full"]].drop_duplicates()

# Merge (small × small)
df_lookup = pd.merge(df_articles, df_sport,left_on="qid",right_on="QID",how="left")
df_lookup.head()

,date,qid,article,pageviews,language_full,predicted_label,QID,score,truth
0,2024-06-26,Q1,Universo,90,Spanish,Non-Sport,Q1,0.847515,NS
1,2024-06-26,Q1,Univers,98,French,Non-Sport,Q1,0.847515,NS
2,2024-06-26,Q1,Universo,141,Portuguese,Non-Sport,Q1,0.847515,NS
3,2024-06-26,Q1,Universe,98,English,Non-Sport,Q1,0.847515,NS
4,2024-06-26,Q1,宇宙,360,Japanese,Non-Sport,Q1,0.847515,NS


In [86]:
len(df_lookup)

7442858

In [87]:
df_lookup.drop(columns=['QID'], inplace=True)
df_lookup.head()

,date,qid,article,pageviews,language_full,predicted_label,score,truth
0,2024-06-26,Q1,Universo,90,Spanish,Non-Sport,0.847515,NS
1,2024-06-26,Q1,Univers,98,French,Non-Sport,0.847515,NS
2,2024-06-26,Q1,Universo,141,Portuguese,Non-Sport,0.847515,NS
3,2024-06-26,Q1,Universe,98,English,Non-Sport,0.847515,NS
4,2024-06-26,Q1,宇宙,360,Japanese,Non-Sport,0.847515,NS


In [88]:
periods = {
    "Before": ("2024-06-26", "2024-07-25"),
    "During": ("2024-07-26", "2024-08-11"),
    "After":  ("2024-08-12", "2024-09-11")
}

In [89]:
def assign_period(date):
    for name, (start, end) in periods.items():
        if pd.to_datetime(start) <= date <= pd.to_datetime(end):
            return name
    return "Other"

In [90]:
df_lookup['date'] = pd.to_datetime(df_lookup['date'])

In [91]:
df_lookup['period'] = "Other"

df_lookup.loc[
    (df_lookup['date'] >= "2024-06-26") & (df_lookup['date'] <= "2024-07-25"),
    'period'
] = "Before"

df_lookup.loc[
    (df_lookup['date'] >= "2024-07-26") & (df_lookup['date'] <= "2024-08-11"),
    'period'
] = "During"

df_lookup.loc[
    (df_lookup['date'] >= "2024-08-12") & (df_lookup['date'] <= "2024-09-11"),
    'period'
] = "After"


In [92]:
df_lookup.head()

,date,qid,article,pageviews,language_full,predicted_label,score,truth,period
0,2024-06-26,Q1,Universo,90,Spanish,Non-Sport,0.847515,NS,Before
1,2024-06-26,Q1,Univers,98,French,Non-Sport,0.847515,NS,Before
2,2024-06-26,Q1,Universo,141,Portuguese,Non-Sport,0.847515,NS,Before
3,2024-06-26,Q1,Universe,98,English,Non-Sport,0.847515,NS,Before
4,2024-06-26,Q1,宇宙,360,Japanese,Non-Sport,0.847515,NS,Before


In [93]:
df_lookup.to_csv("final.csv",index=False)

In [94]:
def load_all_data(url):
    df_all = pd.read_csv(url,skiprows=0)  # adjust skiprows if needed
    df_all.columns = df_all.columns.str.strip()
    
    # Optional: lower-case all column names
    df_all.columns = df_all.columns.str.lower()

    df_all["article"] = df_all["article"].astype(str)
    df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
    return df_all

csv_url = "https://drive.google.com/uc?export=download&id=12pniP3TvqHcICKPMrwFMYbXwAxvHNS1j"
df_all = load_all_data(csv_url)

KeyError: 'article'

In [95]:
import pandas as pd

csv_url = "https://drive.google.com/uc?export=download&id=12pniP3TvqHcICKPMrwFMYbXwAxvHNS1j"
df_all = pd.read_csv(csv_url)

# Show all column names
print(df_all.columns.tolist())


['<!DOCTYPE html><html><head><title>Google Drive - Virus scan warning</title><meta http-equiv="content-type" content="text/html; charset=utf-8"/><style nonce="9ew-U8pToNEi2rr21UH3Bg">.goog-link-button{position:relative;color:#15c;text-decoration:underline;cursor:pointer}.goog-link-button-disabled{color:#ccc;text-decoration:none;cursor:default}body{color:#222;font:normal 13px/1.4 arial', 'sans-serif;margin:0}.grecaptcha-badge{visibility:hidden}.uc-main{padding-top:50px;text-align:center}#uc-dl-icon{display:inline-block;margin-top:16px;padding-right:1em;vertical-align:top}#uc-text{display:inline-block;max-width:68ex;text-align:left}.uc-error-caption', '.uc-warning-caption{color:#222;font-size:16px}#uc-download-link{text-decoration:none}.uc-name-size a{color:#15c;text-decoration:none}.uc-name-size a:visited{color:#61c;text-decoration:none}.uc-name-size a:active{color:#d14836;text-decoration:none}.uc-footer{color:#777;font-size:11px;padding-bottom:5ex;padding-top:5ex;text-align:center}.uc-